# Informe Técnico del Proyecto

## TFM - Políticas de salud pública

Este informe resume el trabajo realizado en el proyecto, explicando de forma secuencial el objetivo, las decisiones metodológicas, los criterios de depuración de datos, las reducciones de dimensionalidad aplicadas, la integración de fuentes, el clustering, la regresión multivariante con SHAP y el análisis final de recomendaciones para el cluster más vulnerable.

## 1. Objetivo del proyecto

El objetivo del TFM fue analizar cómo distintos indicadores de salud pública y contexto socioeconómico se relacionan con la esperanza de vida y con los patrones de mortalidad entre países. En concreto, el proyecto integró información sobre:

- cobertura vacunal
- esperanza de vida
- gasto público en salud como porcentaje del PIB
- PIB per cápita y población
- tasa de suicidios
- causas de muerte, con foco en causas infecciosas

A partir de estas fuentes se construyó una secuencia analítica con cinco metas:

1. depurar y homogeneizar datasets internacionales con estructura país-año;
2. estudiar correlaciones relevantes mediante EDA y análisis bivariado;
3. sintetizar información redundante con PCA cuando varias variables medían un mismo fenómeno;
4. segmentar países en perfiles sanitarios comparables mediante clustering;
5. estimar el peso de cada variable sobre la esperanza de vida con una regresión lineal multivariante e interpretarla con SHAP.


## 2. Flujo general de trabajo

El pipeline del proyecto se desarrolló en el siguiente orden:

1. limpieza y análisis exploratorio de vacunación y esperanza de vida;
2. análisis de gasto público en salud y su relación con esperanza de vida;
3. análisis de suicidios y esperanza de vida;
4. construcción de un indicador sintético de carga infecciosa mediante PCA sobre tasas de mortalidad;
5. merge de los distintos bloques de información en un dataset final país-año;
6. clustering de países con base en variables sanitarias clave;
7. regresión lineal multivariante para modelar esperanza de vida;
8. interpretación global y por cluster con SHAP;
9. recomendaciones específicas para el cluster con mayor vulnerabilidad infecciosa.


## 3. EDA de vacunación y esperanza de vida

### 3.1. Criterios de limpieza

En el notebook `Limpieza_y_PCA_vacunas_mortalidad.ipynb` se trabajó primero con cobertura vacunal y esperanza de vida. Los criterios aplicados fueron:

- exclusión de agregados regionales y entidades no comparables con países;
- restricción temporal al periodo 2000-2019 para alinear fuentes;
- selección de variables vacunales básicas con cobertura completa;
- descarte de vacunas con porcentajes elevados de nulos para evitar imputaciones artificiales.

La decisión metodológica más importante fue conservar únicamente `DTP3`, `MCV1` y `Pol3`, porque eran variables con cobertura prácticamente completa y además representan indicadores estándar del calendario básico de inmunización.

El dataset resultante de esta etapa quedó en **3.841 observaciones**, con **193 países** y el periodo **2000-2019**.

### 3.2. Correlaciones entre vacunación y esperanza de vida

Se utilizó correlación de Pearson para medir asociación lineal entre coberturas vacunales y esperanza de vida. Los resultados fueron consistentes y positivos:

| Variable | Correlación con esperanza de vida |
|---|---:|
| DTP3 | 0.623 |
| MCV1 | 0.620 |
| Pol3 | 0.626 |

La lectura de estos resultados es clara: a mayor cobertura vacunal básica, mayor esperanza de vida media observada. No implica causalidad directa por sí sola, pero sí apoya la idea de que la vacunación funciona como un buen proxy de fortaleza del sistema sanitario y de prevención.

![Matriz de correlación vacunación-vida](../Notebooks/visualizaciones/1%20Limpieza_y_PCA_vacunas_mortalidad/cell_28_visual_02.png)

### 3.3. PCA de vacunación

Como `DTP3`, `MCV1` y `Pol3` estaban muy correlacionadas entre sí, se aplicó un **PCA con una sola componente** para sintetizar la información y evitar redundancia en etapas posteriores.

Resultados principales:

- la primera componente explicó **95,5%** de la varianza total;
- la correlación entre la componente PCA de vacunación y la esperanza de vida fue **0,638**, ligeramente superior a la de cada vacuna por separado.

Esto justificó sustituir tres variables muy solapadas por un único indicador sintético: **`Componente PCA (vacunas)`**. El criterio fue conservar la mayor parte de la información con una representación más compacta y más estable para clustering y regresión.

## 4. EDAs adicionales y análisis de correlación

### 4.1. Gasto público en salud y esperanza de vida

En el notebook `EDA gasto en salud-pbi.ipynb` se depuró el dataset de gasto público en salud eliminando registros sin código país y verificando duplicados, cobertura temporal y consistencia de tipos. Posteriormente se integró con esperanza de vida para trabajar con observaciones comparables país-año.

En una primera aproximación global, el análisis mostró una **correlación positiva moderada de 0,506** entre gasto público en salud (% del PIB) y esperanza de vida, sobre **3.754 observaciones** válidas.

Sin embargo, el notebook no se limitó a esa lectura agregada. También se realizó una **segmentación por PIB per cápita**, aislando a los países del **cuartil más alto** para comparar el comportamiento del gasto en sistemas relativamente más ricos. El criterio fue metodológicamente útil porque permite observar si, incluso entre economías con mayor capacidad material, las diferencias en prioridad sanitaria siguen asociándose con diferencias en resultados.

Además, dentro de ese grupo de países con PIB per cápita alto se aplicó una **ponderación por población** al calcular la evolución anual de gasto y esperanza de vida. Esta decisión evitó tratar igual a países muy pequeños y a países mucho más poblados, dando más peso a los casos con mayor volumen de población expuesta.

Con esa estrategia, la **correlación ponderada entre gasto en salud y esperanza de vida ascendió a 0,865**, lo que constituye una relación fuerte. Este resultado refuerza la idea de que, cuando se controla parcialmente por nivel de ingreso y se considera el peso demográfico, la inversión sanitaria pública muestra una asociación mucho más nítida con los resultados de salud.

![Correlación ponderada entre gasto en salud y esperanza de vida](../Notebooks/visualizaciones/1%20EDA_gasto_en_salud_pbi/cell_80_visual_09.png)

Metodológicamente, este bloque era importante porque añadía una dimensión de capacidad de inversión pública al análisis. La lectura final no es simplemente que “más gasto implica siempre mejores resultados”, sino que el gasto parece ser especialmente relevante cuando se observa en contextos comparables de renta y cuando se pondera adecuadamente por tamaño poblacional.

![Evolución ponderada de gasto en salud y esperanza de vida en países con PIB per cápita alto](../Notebooks/visualizaciones/1%20EDA_gasto_en_salud_pbi/cell_79_visual_08.png)

El gráfico anterior resume bien esta parte del análisis: en los países del cuartil alto de PIB per cápita, la trayectoria del gasto en salud ponderado y la esperanza de vida ponderada evoluciona de forma claramente acompasada a lo largo del tiempo.

### 4.2. Suicidios y esperanza de vida

En `suicidios.ipynb` se unieron las tasas de suicidio estandarizadas por edad con la serie de esperanza de vida. El resultado fue una **correlación de -0,379** sobre **3.680 observaciones**.

La interpretación es que, en promedio, los contextos con peor desempeño sanitario y social tienden a presentar también mayor carga de suicidio. La relación es negativa y de intensidad intermedia, por lo que se utilizó como evidencia complementaria, no como eje estructural del modelo final.

## 5. Mortalidad por causas y PCA de carga infecciosa

Una parte clave del proyecto fue transformar el dataset de causas de muerte en un bloque analítico comparable entre países. Para ello se construyeron **tasas por 100.000 habitantes** y se evaluaron correlaciones entre causas, en vez de trabajar con totales absolutos, que están muy condicionados por el tamaño poblacional.

La matriz `correlacao_causas` permitió ver qué causas se movían conjuntamente y detectar un bloque infeccioso bien definido.

![Heatmap de correlación entre causas de muerte](../Notebooks/visualizaciones/1%20Limpieza_y_PCA_vacunas_mortalidad/correlacao_causas.png)

A partir de ahí se seleccionaron nueve variables infecciosas para construir un PCA específico:

- infecciones respiratorias bajas
- tuberculosis
- VIH/SIDA
- malaria
- enfermedades diarreicas
- trastornos maternos
- trastornos neonatales
- meningitis
- hepatitis aguda

La lógica fue agrupar causas fuertemente relacionadas con entornos de baja cobertura sanitaria, alta fragilidad preventiva y peores condiciones de saneamiento.

La primera componente del PCA infeccioso explicó **68,6%** de la varianza, suficiente para representar la carga infecciosa agregada sin perder la señal principal.

Las variables con mayor peso absoluto en la componente fueron:

| Variable | Loading |
|---|---:|
| Enfermedades diarreicas | 0.376 |
| Trastornos neonatales | 0.375 |
| Trastornos maternos | 0.370 |
| Meningitis | 0.369 |
| Infecciones respiratorias bajas | 0.356 |

Esta componente se guardó como **`Infectious_mortality_PC1`** y pasó a representar una dimensión estructural del riesgo sanitario.

## 6. Merge de fuentes y construcción del dataset final

En `Merge_y_Clustering.ipynb` se integraron las distintas piezas del proyecto. El merge se hizo sobre las claves **`Code` + `Year`**, partiendo del dataset limpio de esperanza de vida y añadiendo, mediante joins sucesivos:

- el PCA de mortalidad infecciosa y tasas asociadas;
- la componente PCA de vacunación;
- gasto público en salud, PIB per cápita y población.

Se seleccionaron solo las columnas necesarias para evitar arrastrar variables redundantes o no alineadas entre fuentes. El resultado final para modelización fue `df_para_regresion.csv`, con **3.800 filas**, **190 países** y el periodo completo **2000-2019**.

Este paso fue importante porque transformó varios análisis parciales en una única tabla país-año lista para segmentación y modelización.

## 7. Clustering de países

### 7.1. Variables utilizadas

El clustering se realizó a nivel país, promediando en el tiempo cuatro dimensiones principales:

- esperanza de vida
- componente PCA de vacunación
- gasto público en salud (% del PIB)
- componente PCA de mortalidad infecciosa

Estas variables fueron estandarizadas con `StandardScaler` para evitar que la escala numérica de unas dominara sobre otras.

### 7.2. Evaluación del número de clusters

Se compararon soluciones con **K=2, K=3 y K=4** mediante dos métricas internas:

| K | Silhouette | Davies-Bouldin |
|---|---:|---:|
| 2 | 0.376 | 1.064 |
| 3 | 0.402 | 0.919 |
| 4 | 0.416 | 0.847 |

![Comparación de métricas de clustering](../Notebooks/visualizaciones/2%20Merge_y_Clustering/cell_76_visual_02.png)

Aunque **K=4** mejoró ligeramente las métricas internas, se optó por trabajar con **3 clusters** en el dataset final porque ofrecía un equilibrio mejor entre separación estadística, estabilidad interpretativa y utilidad narrativa para el TFM. En la práctica, K=4 tendía a subdividir perfiles de alto desempeño sin aportar una mejora conceptual proporcional.

### 7.3. Resultado e interpretación de los clusters

La solución final generó tres perfiles:

| Cluster | Países | Esperanza de vida | Vacunación PCA | Gasto salud % PIB | Carga infecciosa |
|---|---:|---:|---:|---:|---:|
| Alta carga infecciosa / Menor esperanza de vida | 44 | 57.97 | -2.02 | 1.26 | 2.82 |
| Transición sanitaria / Esperanza de vida intermedia | 91 | 71.06 | 0.53 | 2.65 | -0.98 |
| Alta esperanza de vida / Alta inversión y prevención | 55 | 76.93 | 0.77 | 5.74 | -1.59 |

La lectura sustantiva es muy clara:

- el **cluster 0** agrupa países con menor esperanza de vida, menor cobertura vacunal, menor esfuerzo público en salud y alta mortalidad infecciosa;
- el **cluster 1** representa una transición sanitaria intermedia, con mejora de indicadores pero sin alcanzar todavía el perfil de mayor resiliencia;
- el **cluster 2** concentra sistemas con mejor desempeño preventivo, mayor inversión y menor peso de enfermedades infecciosas.

![Distribución de los 3 clusters en el espacio PCA](../Notebooks/visualizaciones/2%20Merge_y_Clustering/cell_80_visual_04.png)

## 8. Regresión lineal multivariante

En `Regresión_y_SHAP.ipynb` se ajustó una regresión lineal multivariante para predecir la esperanza de vida a partir de cinco variables:

- `Infectious_mortality_PC1`
- `Cardiovascular diseases_rate`
- `Neoplasms_rate`
- `Componente PCA (vacunas)`
- `Domestic general government health expenditure (% of GDP)`

El corte temporal se hizo con una lógica de predicción hacia adelante: entrenamiento hasta **2015** y test en **2016-2019**.

### 8.1. Rendimiento global del modelo

| Split | RMSE | MAE | R² |
|---|---:|---:|---:|
| Train | 3.67 | 2.65 | 0.841 |
| Test | 3.53 | 2.77 | 0.787 |

El ajuste global fue bueno: el modelo explicó cerca del **79%** de la variabilidad en datos futuros y mantuvo un error absoluto medio de alrededor de **2,8 años** de esperanza de vida. Esto sugiere que el conjunto de variables seleccionadas capturó una parte sustancial de la estructura sanitaria comparada.

## 9. Interpretación del modelo con SHAP

La interpretación con SHAP permitió ir más allá del signo de los coeficientes y cuantificar qué variables movían realmente la predicción.

### 9.1. Importancia global

| Variable | Coeficiente | mean(|SHAP|) |
|---|---:|---:|
| Mortalidad infecciosa | -3.074 | 3.853 |
| Neoplasias | 0.040 | 3.373 |
| Mortalidad cardiovascular | -0.013 | 1.823 |
| Vacunación (PCA) | 0.273 | 0.285 |
| Gasto salud (% PIB) | 0.000 | 0.001 |

La variable con mayor impacto global fue claramente **la mortalidad infecciosa**, lo que encaja con toda la narrativa anterior del proyecto: cuando la carga infecciosa aumenta, la esperanza de vida esperada cae con fuerza. Las neoplasias aparecieron también como muy influyentes, en este caso asociadas a perfiles epidemiológicos más avanzados y longevos.

![Importancia global de variables según SHAP](../Notebooks/visualizaciones/3%20Regresion_y_SHAP/cell_24_visual_03.png)

### 9.2. Modelo específico del cluster 0

Dado que el cluster 0 representaba el grupo más vulnerable, se ajustó un modelo específico para ese segmento.

- observaciones de entrenamiento: **704**
- observaciones de test: **176**
- países incluidos: **44**

| Modelo cluster 0 | RMSE test | MAE test | R² test |
|---|---:|---:|---:|
| Cluster 0 | 3.527 | 2.785 | 0.266 |

El ajuste cayó con claridad respecto al modelo global. Esto es interpretable: dentro del cluster más frágil hay más heterogeneidad estructural, mayor volatilidad y más factores no observados. Aun así, SHAP volvió a señalar a la **mortalidad infecciosa** como variable dominante, seguida por mortalidad cardiovascular, gasto en salud y vacunación.

La conclusión técnica es que el cluster 0 es más difícil de modelar con una relación lineal simple, precisamente porque concentra contextos sanitarios más complejos y menos estables.

## 10. Análisis del notebook de recomendaciones

El notebook `Recomendaciones_cluster_0.ipynb` se centra en el **cluster 0**, definido como el grupo con **alta carga infecciosa y menor esperanza de vida**. En esta etapa se buscó responder dos preguntas:

1. ¿qué enfermedades infecciosas explican mejor la vulnerabilidad del cluster?
2. ¿qué medidas de política pública tienen más sentido dadas esas prioridades?

### 10.1. Enfermedades más relevantes del cluster 0

Cuando se ordenan las tasas medias de mortalidad por 100.000 habitantes, las causas más importantes son:

| Enfermedad | Tasa media |
|---|---:|
| VIH/SIDA | 115.6 |
| Trastornos neonatales | 91.0 |
| Infecciones respiratorias bajas | 86.3 |
| Enfermedades diarreicas | 85.0 |
| Malaria | 73.3 |
| Tuberculosis | 53.2 |

![Enfermedades infecciosas más letales en el cluster 0](../Notebooks/visualizaciones/4%20Recomendaciones_cluster_0/cell_17_visual_01.png)

Si se mira el impacto absoluto acumulado, destacan especialmente los **trastornos neonatales**, las **enfermedades diarreicas** y las **infecciones respiratorias bajas**, lo que sugiere que el problema no es únicamente epidemiológico, sino también de acceso temprano, salud materno-infantil, nutrición y saneamiento.

### 10.2. Países con mayor carga infecciosa total

El análisis agregado por país mostró que la carga infecciosa total del cluster 0 se concentra con especial intensidad en:

- Lesotho
- Central African Republic
- Eritrea
- Zimbabwe
- Mozambique
- Niger
- Sierra Leone
- Chad

![Top 10 países con mayor carga infecciosa total](../Notebooks/visualizaciones/4%20Recomendaciones_cluster_0/cell_22_visual_03.png)

Este resultado refuerza la necesidad de interpretar el cluster 0 como un perfil sanitario coherente, no como un conjunto de casos aislados.

### 10.3. Medidas recomendadas

Las recomendaciones derivadas del análisis se organizaron en torno a las enfermedades y cuellos de botella más relevantes:

- **VIH/SIDA**: ampliar diagnóstico temprano, tratamiento antirretroviral y estrategias focalizadas de prevención.
- **Trastornos neonatales y maternos**: reforzar atención prenatal, parto seguro, unidades neonatales básicas y seguimiento del recién nacido.
- **Infecciones respiratorias bajas**: mejorar vacunación, cobertura de atención primaria, detección precoz y acceso a antibióticos cuando corresponda.
- **Enfermedades diarreicas**: invertir en agua segura, saneamiento, higiene comunitaria y manejo de deshidratación.
- **Malaria**: fortalecer control vectorial, mosquiteras impregnadas, quimioprofilaxis y respuesta territorial.
- **Tuberculosis**: ampliar cribado, diagnóstico microbiológico y adherencia terapéutica.

En términos de política pública, el hallazgo más importante es que el cluster 0 necesita un enfoque más integral que exclusivamente hospitalario: prevención, salud comunitaria, vacunación, salud materno-infantil y saneamiento básico aparecen como pilares de mayor retorno esperado.

## 11. Conclusiones generales

El proyecto permitió construir una narrativa analítica consistente desde el dato bruto hasta la recomendación final.

Principales conclusiones:

- la vacunación básica mostró una relación positiva robusta con la esperanza de vida y pudo sintetizarse eficazmente con PCA;
- el gasto público en salud también se relacionó positivamente con la esperanza de vida, aunque con menor intensidad que la dimensión preventiva;
- la tasa de suicidios mostró una relación negativa intermedia, útil como evidencia complementaria del contexto sanitario y social;
- la carga de mortalidad infecciosa emergió como la dimensión más determinante del proyecto, tanto en el clustering como en la regresión y en SHAP;
- los tres clusters permitieron traducir un espacio continuo de desigualdades sanitarias en perfiles interpretables y accionables;
- el cluster 0 concentró la mayor vulnerabilidad estructural, justificando un análisis específico de enfermedades prioritarias y medidas concretas.

En conjunto, el trabajo confirma que la esperanza de vida no depende de un único indicador, sino de una arquitectura sanitaria amplia donde prevención, inversión, estructura epidemiológica y capacidad del sistema interactúan entre sí.